# Initial Route Set Generation

In [1]:
%load_ext autoreload
%reload_ext autoreload

import copy
import multiprocessing as mp
import os
import random
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

import geopandas as gpd
import networkx as nx
import osmnx as ox
import utm
from networkx import MultiDiGraph
from shapely.geometry import LineString

from config.config import (
    FILTER_OPTIONS,
    MAX_DISTANCE,
    N_NETWORKS,
    N_ROUTES,
    STOP_MIN,
    WALKING_DISTANCES,
    WD_TO_USE,
)
from genetic_algorithm.reverse_routes import get_reverse_route
from genetic_algorithm.schema import Network, Route, Stop
from utils.checkpoint_utils import load_data, save_data
from utils.graph_utils import generate_graph, load_graph, save_route_graph_to_plot
from utils.route_utils import (
    add_points_to_graph,
    disable_surrounding_nodes,
    enable_nodes,
    get_enabled_node_with_highest_edge_probability,
    haversine,
    make_run_tag,
    meters_to_degrees,
)


## Graph and Features

In [2]:
# NOTE: SELECT THE CITY HERE, COMMENT OUT THE REMAINING CITIES
CITY_NAME = "Mandaluyong" # Taguig, Mandaluyong, Manila, Makati
COUNTRY = "Philippines"
MODE = "drive"

LOCATION = f"{CITY_NAME}, {COUNTRY}"
TARGET_MAP_DIR = "../data/street_networks"
GRAPH_DIR = os.path.join(TARGET_MAP_DIR, f"{CITY_NAME}.graphml")

if os.path.exists(GRAPH_DIR):
    CITY_GRAPH = load_graph(GRAPH_DIR)
else:
    CITY_GRAPH = generate_graph(LOCATION, MODE, GRAPH_DIR)

# Iterate over the edges and assign a unique edge_id
for edge_id, (u, v, k) in enumerate(CITY_GRAPH.edges(keys=True)):
    CITY_GRAPH[u][v][k]["edge_id"] = edge_id

Graph loaded successfully
NUMBER OF EDGES:  2452
NUMBER OF NODES:  1024


## Initial Filtering

In [3]:
### For Filtering the roads and other features

# GETTING ROADS AND WATERWAYS
road = ox.graph_to_gdfs(CITY_GRAPH, nodes=False, edges=True)

# Get all the roads that are not junctions (ex. Roundabouts, intersection, etc.)
filtered_roads = road[road['junction'].isna()]

# Separate roads whose highway types are only one value and those that are more than 1 (lists)
rows_with_lists = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, list))]
rows_with_strings = filtered_roads[filtered_roads['highway'].apply(lambda x: isinstance(x, str))]

# Allowed Roads to place stops
filter_options = FILTER_OPTIONS

# To separate zones
separation_options = ['primary', 'secondary', 'tertiary', 'unclassified']

# Get the roads whose widths are above the threshold
def check_list(lst):
    return any(x in filter_options for x in lst)

# Download OpenStreetMap data for the area of interest
waterways = ox.features_from_place(LOCATION, tags={'waterway': True})
filtered_rivers = waterways[waterways['waterway'].isin(['river'])]
filtered_streams = waterways[waterways['waterway'].isin(['stream'])]

# Get all the roads with the allowed road types
filtered_roads_strings = rows_with_strings.loc[rows_with_strings['highway'].isin(filter_options)]
filtered_roads_lists = rows_with_lists[rows_with_lists['highway'].apply(check_list)]

# Create spatial index
filtered_roads_strings_sindex = filtered_roads_strings.sindex
filtered_roads_lists_sindex = filtered_roads_lists.sindex
filtered_rivers_sindex = filtered_rivers.sindex
filtered_streams_sindex = filtered_streams.sindex

# Total Distance of the allowed road types
TOTAL_ROAD_NETWORK_DISTANCE = 0
for index, row in filtered_roads_strings.iterrows():
    TOTAL_ROAD_NETWORK_DISTANCE += row['length']

## Stop Placement


In [4]:
def _linestring_meters(line):
    coords = list(line.coords)
    return sum(
        haversine((coords[i][1], coords[i][0]), (coords[i + 1][1], coords[i + 1][0])) * 1000.0
        for i in range(len(coords) - 1)
    )


def place_stops_on_roads(amenity_graph, graph_of_stops, list_of_stops):
    list_relevant_edges_set = set()
    string_relevant_edges_set = set()

    stop_id = 0
    for n, d in amenity_graph.nodes(data=True):
        assert 'geometry' in d and d['geometry'] is not None, f"Missing geometry for amenity node {n}"

    node_sizes = {
        n: a for n, a in zip(
            amenity_graph.nodes,
            gpd.GeoSeries([d['geometry'] for _, d in amenity_graph.nodes(data=True)],
                          crs="EPSG:4326").to_crs(32651).area,
        )
    }
    print(f"Starting stop placement for {amenity_graph.number_of_nodes()} amenity nodes...")

    for node_key, node_data in amenity_graph.nodes(data=True):
        geom = node_data['geometry']

        if geom.geom_type in ['Point']:
            if node_data.get('amenity') == 'transportation':
                assert ('x' in node_data and 'y' in node_data) or not geom.is_empty, (
                    f"Missing coordinates for transportation point {node_key}"
                )
                px = node_data.get('x', geom.x)
                py = node_data.get('y', geom.y)

                nearest_edge = ox.distance.nearest_edges(graph_of_stops, X=px, Y=py, return_dist=False)
                node1, node2, key = nearest_edge
                assert graph_of_stops.has_edge(node1, node2, key), (
                    f"Nearest edge ({node1}, {node2}, {key}) not found in graph_of_stops"
                )
                data = graph_of_stops.edges[node1, node2, key]

                if 'geometry' not in data:
                    gdf_edges = ox.graph_to_gdfs(graph_of_stops, nodes=False, edges=True)
                    assert nearest_edge in gdf_edges.index, f"Edge {nearest_edge} missing from gdf_edges"
                    line = gdf_edges.loc[nearest_edge]['geometry']
                    edge_length = gdf_edges.loc[nearest_edge]['length']
                else:
                    line = data['geometry']
                    edge_length = data['length']

                assert len(line.coords) >= 2, f"Invalid LineString with fewer than 2 coords on edge ({node1}, {node2}, {key})"

                if edge_length > 20:
                    assert node1 in graph_of_stops.nodes and node2 in graph_of_stops.nodes, (
                        f"Endpoint node missing in graph_of_stops: node1={node1}, node2={node2}"
                    )
                    assert 'y' in graph_of_stops.nodes[node1] and 'x' in graph_of_stops.nodes[node1], (
                        f"Missing x/y on node1={node1}"
                    )
                    assert 'y' in graph_of_stops.nodes[node2] and 'x' in graph_of_stops.nodes[node2], (
                        f"Missing x/y on node2={node2}"
                    )

                    print(f"[Transpo Stop {stop_id}] Amenity={node_key} | Edge=({node1}, {node2}, {key}) | Length={edge_length:.2f}m")
                    if node1 < 10000:
                        print(f"  -> NOTICE: node1 {node1} is an existing stop ({graph_of_stops.nodes[node1]['y']}, {graph_of_stops.nodes[node1]['x']})")
                    if node2 < 10000:
                        print(f"  -> NOTICE: node2 {node2} is an existing stop ({graph_of_stops.nodes[node2]['y']}, {graph_of_stops.nodes[node2]['x']})")

                    point_a = (graph_of_stops.nodes[node1]['y'], graph_of_stops.nodes[node1]['x'])
                    point_c = (graph_of_stops.nodes[node2]['y'], graph_of_stops.nodes[node2]['x'])

                    if len(line.coords) == 2:
                        random_position = random.uniform(0.3 * edge_length, edge_length * 0.7)
                        node1_utm = utm.from_latlon(point_a[0], point_a[1])
                        node2_utm = utm.from_latlon(point_c[0], point_c[1])
                        utm_line = LineString([(node1_utm[0], node1_utm[1]), (node2_utm[0], node2_utm[1])])

                        point_on_road_utm = calculate_coordinate_along_edge(utm_line, random_position)
                        to_convert = (point_on_road_utm[0], point_on_road_utm[1], node1_utm[2], node1_utm[3])
                        point_on_road = utm.to_latlon(*to_convert)
                        lon = point_on_road[1]
                        lat = point_on_road[0]
                    else:
                        coordinates = list(line.coords)
                        split_idx = random.randint(1, len(coordinates) - 2)
                        point_on_road = coordinates[split_idx]
                        lon = point_on_road[0]
                        lat = point_on_road[1]

                    isTranspo = True
                    assert stop_id not in graph_of_stops.nodes, f"Duplicate stop_id {stop_id} in graph_of_stops"
                    graph_of_stops.add_node(stop_id, x=lon, y=lat, isTranspo=isTranspo)
                    list_of_stops.append(Stop(lat, lon, True, stop_id, 0))

                    point_b = (lat, lon)

                    if len(line.coords) == 2:
                        line_ab = LineString([(point_a[1], point_a[0]), (point_b[1], point_b[0])])
                        line_bc = LineString([(point_b[1], point_b[0]), (point_c[1], point_c[0])])
                    else:
                        line_ab = LineString(coordinates[:split_idx + 1])
                        line_bc = LineString(coordinates[split_idx:])

                    distance_ab = _linestring_meters(line_ab)
                    distance_bc = _linestring_meters(line_bc)

                    print(f"  -> Placed at ({lat:.6f}, {lon:.6f}) | Split lengths: AB={distance_ab:.2f}m, BC={distance_bc:.2f}m (orig={edge_length:.2f}m)")

                    edge_data = data.copy()
                    edge_data['length'] = distance_ab
                    edge_data['geometry'] = line_ab
                    graph_of_stops.add_edge(node1, stop_id, **edge_data)

                    edge_data = data.copy()
                    edge_data['length'] = distance_bc
                    edge_data['geometry'] = line_bc
                    graph_of_stops.add_edge(stop_id, node2, **edge_data)

                    assert graph_of_stops.has_edge(node1, stop_id) and graph_of_stops.has_edge(stop_id, node2), (
                        f"Failed adding split edges for stop {stop_id}"
                    )

                    graph_of_stops.remove_edge(node1, node2, key=key)
                    stop_id += 1
                else:
                    print(f"[Skip Transpo] Amenity={node_key} | Edge=({node1}, {node2}, {key}) too short ({edge_length:.2f}m <= 20m)")

        elif geom.geom_type in ['MultiPolygon', 'Polygon']:
            buffer_poly = geom.buffer(meters_to_degrees(30))
            relevant_edges = get_relevant_edges(buffer_poly, list_relevant_edges_set, string_relevant_edges_set)
            assert node_key in node_sizes, f"Missing node_key {node_key} in node_sizes"
            node_size = node_sizes[node_key]

            print(f"[Zone {node_key}] Area={node_size:.1f}m² | Candidate edges={len(relevant_edges)}")

            if CITY_NAME == "Taguig":
                is_transpo_zone = bool(node_data.get('isTranspo', False))
                stop_id = place_stops_along_edges(relevant_edges, buffer_poly, node_size, graph_of_stops, list_of_stops, stop_id, is_transpo_zone)
            else:
                stop_id = place_stops_along_edges(relevant_edges, buffer_poly, node_size, graph_of_stops, list_of_stops, stop_id)

    print(f"Completed. Total stops placed: {stop_id}")
    return graph_of_stops, list_of_stops


def get_relevant_edges(polygon, list_relevant_edges_set, string_relevant_edges_set):
    relevant_edges = []

    possible_matches_roads = filtered_roads_strings.iloc[list(filtered_roads_strings_sindex.intersection(polygon.bounds))]
    for index, row in possible_matches_roads.iterrows():
        assert 'geometry' in row and 'highway' in row and 'edge_id' in row, (
            f"Missing required column in filtered_roads_strings at index {index}"
        )
        if polygon.intersects(row['geometry']) and row['highway'] in filter_options:
            row_edge_id = row['edge_id']
            if row_edge_id not in string_relevant_edges_set:
                relevant_edges.append([index, row])
                string_relevant_edges_set.add(row_edge_id)

    possible_matches_lists = filtered_roads_lists.iloc[list(filtered_roads_lists_sindex.intersection(polygon.bounds))]
    for index, row in possible_matches_lists.iterrows():
        assert 'geometry' in row and 'highway' in row and 'edge_id' in row, (
            f"Missing required column in filtered_roads_lists at index {index}"
        )
        if polygon.intersects(row['geometry']):
            list_highway = row['highway']
            if any(x in filter_options for x in list_highway):
                row_edge_id = row['edge_id']
                if row_edge_id not in list_relevant_edges_set:
                    relevant_edges.append([index, row])
                    list_relevant_edges_set.add(row_edge_id)

    return relevant_edges


def place_stops_along_edges(edges, polygon, node_size, graph_of_stops, list_of_stops, stop_id, is_transpo_zone=None):
    if len(edges) > 0:
        for edge in edges:
            index = edge[0]
            row = edge[1]
            assert isinstance(index, tuple) and len(index) >= 2, f"Invalid edge index tuple: {index}"
            node1 = index[0]
            node2 = index[1]
            key = index[2] if len(index) > 2 else 0

            if not graph_of_stops.has_edge(node1, node2, key):
                print(f"  -> Skip edge ({node1}, {node2}, {key}): already split or removed from graph_of_stops")
                continue

            data = row.dropna().to_dict() if hasattr(row, 'dropna') else row.copy()

            if 'geometry' not in data:
                gdf_edges = ox.graph_to_gdfs(graph_of_stops, nodes=False, edges=True)
                assert (node1, node2, key) in gdf_edges.index, f"Edge ({node1}, {node2}, {key}) missing in gdf_edges"
                line = gdf_edges.loc[(node1, node2, key)]['geometry']
                edge_length = gdf_edges.loc[(node1, node2, key)]['length']
            else:
                line = data['geometry']
                edge_length = data['length']

            assert len(line.coords) >= 2, f"Invalid LineString with fewer than 2 coords on edge ({node1}, {node2}, {key})"

            if edge_length > 20:
                assert node1 in graph_of_stops.nodes and node2 in graph_of_stops.nodes, (
                    f"Endpoint node missing in graph_of_stops: node1={node1}, node2={node2}"
                )
                assert 'y' in graph_of_stops.nodes[node1] and 'x' in graph_of_stops.nodes[node1], (
                    f"Missing x/y on node1={node1}"
                )
                assert 'y' in graph_of_stops.nodes[node2] and 'x' in graph_of_stops.nodes[node2], (
                    f"Missing x/y on node2={node2}"
                )

                print(f"  [Zone Stop {stop_id}] Edge=({node1}, {node2}, {key}) | Length={edge_length:.2f}m")
                if node1 < 10000:
                    print(f"    -> NOTICE: node1 {node1} is an existing stop ({graph_of_stops.nodes[node1]['y']}, {graph_of_stops.nodes[node1]['x']})")
                if node2 < 10000:
                    print(f"    -> NOTICE: node2 {node2} is an existing stop ({graph_of_stops.nodes[node2]['y']}, {graph_of_stops.nodes[node2]['x']})")

                point_a = (graph_of_stops.nodes[node1]['y'], graph_of_stops.nodes[node1]['x'])
                point_c = (graph_of_stops.nodes[node2]['y'], graph_of_stops.nodes[node2]['x'])

                if len(line.coords) == 2:
                    random_position = random.uniform(0.3 * edge_length, edge_length * 0.7)
                    node1_utm = utm.from_latlon(point_a[0], point_a[1])
                    node2_utm = utm.from_latlon(point_c[0], point_c[1])
                    utm_line = LineString([(node1_utm[0], node1_utm[1]), (node2_utm[0], node2_utm[1])])

                    point_on_road_utm = calculate_coordinate_along_edge(utm_line, random_position)
                    to_convert = (point_on_road_utm[0], point_on_road_utm[1], node1_utm[2], node1_utm[3])
                    point_on_road = utm.to_latlon(*to_convert)
                    lon = point_on_road[1]
                    lat = point_on_road[0]
                else:
                    coordinates = list(line.coords)
                    split_idx = random.randint(1, len(coordinates) - 2)
                    point_on_road = coordinates[split_idx]
                    lon = point_on_road[0]
                    lat = point_on_road[1]

                isTranspo = bool(is_transpo_zone) if is_transpo_zone is not None else False
                assert stop_id not in graph_of_stops.nodes, f"Duplicate stop_id {stop_id} in graph_of_stops"
                graph_of_stops.add_node(stop_id, x=lon, y=lat, isTranspo=isTranspo)
                list_of_stops.append(Stop(lat, lon, isTranspo, stop_id, node_size))

                point_b = (lat, lon)

                if len(line.coords) == 2:
                    line_ab = LineString([(point_a[1], point_a[0]), (point_b[1], point_b[0])])
                    line_bc = LineString([(point_b[1], point_b[0]), (point_c[1], point_c[0])])
                else:
                    line_ab = LineString(coordinates[:split_idx + 1])
                    line_bc = LineString(coordinates[split_idx:])

                distance_ab = _linestring_meters(line_ab)
                distance_bc = _linestring_meters(line_bc)

                print(f"    -> Placed at ({lat:.6f}, {lon:.6f}) | Split lengths: AB={distance_ab:.2f}m, BC={distance_bc:.2f}m (orig={edge_length:.2f}m)")

                edge_data = data.copy()
                edge_data['length'] = distance_ab
                edge_data['geometry'] = line_ab
                graph_of_stops.add_edge(node1, stop_id, **edge_data)

                edge_data = data.copy()
                edge_data['length'] = distance_bc
                edge_data['geometry'] = line_bc
                graph_of_stops.add_edge(stop_id, node2, **edge_data)

                assert graph_of_stops.has_edge(node1, stop_id) and graph_of_stops.has_edge(stop_id, node2), (
                    f"Failed adding split edges for stop {stop_id}"
                )

                graph_of_stops.remove_edge(node1, node2, key=key)
                stop_id += 1
            else:
                print(f"  -> Skip edge ({node1}, {node2}, {key}): too short ({edge_length:.2f}m <= 20m)")

    return stop_id


def calculate_coordinate_along_edge(edge, position):
    assert edge is not None and not edge.is_empty, "Cannot interpolate on empty edge geometry"
    assert 0 <= position <= edge.length, f"Position {position} out of bounds for edge length {edge.length}"
    point = edge.interpolate(position)
    return point.x, point.y

## Route Connection

In [5]:
def generate_route_network(
  stop_nodes: list[Stop],
  max_walking_dist: int,
  max_stops: int,
  max_lines: int,
  graph_of_stops,
  city_area_sum: float,
):
  assert len(stop_nodes) > 0, "stop_nodes cannot be empty"
  assert graph_of_stops is not None and graph_of_stops.number_of_nodes() > 0, (
    "graph_of_stops cannot be empty"
  )
  assert max_lines > 0 and max_stops > 0, "max_lines and max_stops must be positive"

  net_start_time = time.perf_counter()
  for n, data in graph_of_stops.nodes(data=True):
    assert "y" in data and "x" in data, f"Node {n} missing 'y' or 'x' coordinates"
  nodes_yx = {n: (data["y"], data["x"]) for n, data in graph_of_stops.nodes(data=True)}

  overall_graph = nx.MultiDiGraph()
  route_network = []

  num_lines = 0
  max_attempts, attempts = max_lines * 10, 0

  print(f"Target: {max_lines} lines | Candidate stops: {len(stop_nodes)} | Max attempts: {max_attempts}")
  while num_lines < max_lines and attempts < max_attempts:
    attempts += 1
    line_id = num_lines
    forward_route_id = f"{line_id}-F"

    next_nodes = copy.deepcopy(stop_nodes)
    selected_node = random.choice(next_nodes)
    next_nodes.remove(selected_node)

    route_gen, route_length = generate_route(selected_node, next_nodes, max_walking_dist, max_stops, overall_graph, city_area_sum, forward_route_id, graph_of_stops, nodes_yx)

    if len(route_gen) > 1:
      forward_route = Route(route_gen, forward_route_id, route_length, line_id, direction="forward")

      add_to_graph(route_gen, overall_graph, graph_of_stops, forward_route_id, nodes_yx)
      route_network.append(forward_route)

      reverse_route = get_reverse_route(forward_route, graph_of_stops, partner_edges, nodes_yx)

      if reverse_route is not None:
        add_to_graph(
          reverse_route.route,
          overall_graph,
          graph_of_stops,
          reverse_route.route_id,
          nodes_yx,
        )
        route_network.append(reverse_route)
        print(
          f"  └─ Line {line_id} COMPLETE (attempt {attempts}/{max_attempts}): "
          f"Forward ({len(route_gen)} segs, {route_length:.2f} km) + "
          f"Reverse ({len(reverse_route.route)} segs, {reverse_route.length:.2f} km)"
        )
      else:
        print(
          f"  └─ Line {line_id} PARTIAL (attempt {attempts}/{max_attempts}): "
          f"Forward ({len(route_gen)} segs, {route_length:.2f} km), Reverse FAILED"
        )

      num_lines += 1

    else:
      print(f"  [Attempt {attempts}/{max_attempts}] Failed to generate valid forward route for Line {line_id} "
        f"(start stop={selected_node.stop_id}). Retrying...")

  if num_lines < max_lines:
    print(f"  [WARNING] Stopped early: only generated {num_lines}/{max_lines} lines after hitting max_attempts={max_attempts}.")

  net_elapsed = time.perf_counter() - net_start_time
  print(
    f"Finished: {num_lines}/{max_lines} lines ({len(route_network)} directional routes) "
    f"created in {attempts} attempt(s) | Graph: {overall_graph.number_of_nodes()} nodes, "
    f"{overall_graph.number_of_edges()} edges | Done in {net_elapsed:.2f}s."
  )
  return route_network, overall_graph


def add_to_graph(route, overall_graph, graph_of_stops, route_id, nodes_yx):
  assert len(route) > 0, f"Cannot add empty route {route_id} to overall_graph"
  for connection in route:
    assert len(connection) >= 2, (
      f"Invalid connection in route {route_id}: expected >= 2 nodes, got {connection}"
    )
    u_node = connection[0]
    v_node = connection[-1]

    assert u_node in graph_of_stops.nodes, (
      f"Origin node {u_node} not found in graph_of_stops"
    )
    assert v_node in graph_of_stops.nodes, (
      f"Destination node {v_node} not found in graph_of_stops"
    )

    overall_graph.add_node(u_node, **graph_of_stops.nodes[u_node])
    overall_graph.add_node(v_node, **graph_of_stops.nodes[v_node])

    distance_travelled = 0.0
    for i in range(len(connection) - 1):
      seg_u, seg_v = connection[i], connection[i + 1]
      edge_dict = graph_of_stops.get_edge_data(seg_u, seg_v)
      assert edge_dict is not None, f"Missing edge ({seg_u}, {seg_v}) in graph_of_stops"
      edge_data = min(edge_dict.values(), key=lambda d: d.get("length", float("inf")))
      distance_travelled += edge_data["length"] / 1000.0

    overall_graph.add_edge(
      u_node, v_node, key=route_id, road_path=connection, distance=distance_travelled
    )


def generate_route(
  source: Stop, next_nodes: list[Stop], max_walking_dist: int, max_stops: int, network_graph: MultiDiGraph,
  city_area_sum: float, route_id: str, graph_of_stops: nx.MultiDiGraph, nodes_yx: dict,
):
  assert source is not None and source.stop_id in graph_of_stops.nodes, (
    f"Invalid source stop {getattr(source, 'stop_id', None)}"
  )
  assert city_area_sum > 0, f"city_area_sum must be positive, got {city_area_sum}"

  short_route_list = []
  total_distance = 0.0
  orig_node = source
  num_stops = 0
  visited_edges = set()

  max_tries = 20
  current_tries = 0
  fail_reason = "completed_loop"

  while total_distance < MAX_DISTANCE and num_stops < max_stops:
    enable_nodes(next_nodes)
    disable_surrounding_nodes(next_nodes, orig_node, max_walking_dist)
    enabled_nodes = [n for n in next_nodes if n.enabled]
    if len(enabled_nodes) == 0:
      fail_reason = "no_enabled_nodes"
      break

    dest_node = get_enabled_node_with_highest_edge_probability(
      orig_node, enabled_nodes, city_area_sum, network_graph
    )

    if dest_node is None or dest_node.stop_id == orig_node.stop_id:
      fail_reason = "no_valid_dest_node"
      break

    next_nodes.remove(dest_node)

    u_id, v_id = orig_node.stop_id, dest_node.stop_id
    if (u_id, v_id) in visited_edges or (v_id, u_id) in visited_edges:
      current_tries += 1
      if current_tries >= max_tries:
        fail_reason = "max_tries_duplicate_edge"
        break
      continue

    assert v_id in graph_of_stops.nodes, (
      f"Destination stop {v_id} not found in graph_of_stops"
    )
    try:
      shortest_route = nx.shortest_path(
        graph_of_stops, u_id, v_id, weight="length"
      )
    except nx.NetworkXNoPath:
      current_tries += 1
      if current_tries >= max_tries:
        fail_reason = "max_tries_no_path"
        break
      continue

    distance_travelled = 0.0
    for i in range(len(shortest_route) - 1):
      seg_u, seg_v = shortest_route[i], shortest_route[i + 1]
      assert seg_u in nodes_yx and seg_v in nodes_yx, (
        f"Shortest path node missing in nodes_yx: {seg_u} -> {seg_v}"
      )
      edge_dict = graph_of_stops.get_edge_data(seg_u, seg_v)
      assert edge_dict is not None, f"Missing edge ({seg_u}, {seg_v}) in graph_of_stops"
      edge_data = min(edge_dict.values(), key=lambda d: d.get("length", float("inf")))
      distance_travelled += edge_data["length"] / 1000.0

    if total_distance + distance_travelled <= MAX_DISTANCE:
      orig_node.degree += 1
      dest_node.degree += 1
      visited_edges.add((u_id, v_id))
      current_tries = 0

      total_distance += distance_travelled
      short_route_list.append(shortest_route)
      num_stops += 1

      orig_node = dest_node
    else:
      current_tries += 1
      if current_tries >= max_tries:
        fail_reason = f"max_tries_exceeded_max_dist (next_seg={distance_travelled:.2f}km)"
        break
      continue

  min_distance = min(7.0, 0.35 * MAX_DISTANCE)
  min_stops = min(5, max(2, max_stops // 3))

  if len(short_route_list) >= min_stops and total_distance >= min_distance:
    return short_route_list, total_distance
  else:
    print(f"    -> Fail detail: stops={len(short_route_list)}/{min_stops}, dist={total_distance:.2f}/{min_distance:.2f}km | reason={fail_reason}")
    return [], 0.0

In [6]:
AMENITY_MAP_DIR = f"../data/amenity_maps/{CITY_NAME} City"
POLY_FILE = os.path.join(AMENITY_MAP_DIR, f"{CITY_NAME}_polygon.geojson")

merged_amenities_polygons_gdf= gpd.read_file(POLY_FILE)

merged_amenities_polygons_gdf['amenity_points'] = None
merged_amenities_polygons_gdf['area'] = merged_amenities_polygons_gdf.to_crs(epsg=32651).geometry.area

city_area_sum = merged_amenities_polygons_gdf['area'].sum()

In [7]:
SAVED_NETWORKS_DIR = f"../data/saved_networks/{CITY_NAME}"

INITIAL_NETWORK_FILE = os.path.join(SAVED_NETWORKS_DIR, f"{CITY_NAME}_Initial_Network.pkl")
ZONE_NETWORK_FILE = os.path.join(SAVED_NETWORKS_DIR, f"{CITY_NAME}_Zone_Network.pkl")

initial_network = load_data(INITIAL_NETWORK_FILE)
graph_networks_of_polygons = load_data(ZONE_NETWORK_FILE)

add_points_to_graph(initial_network, graph_networks_of_polygons) # Add all points

In [8]:
# LEVEL 3 - CREATING STOPS TO BE PLACED ON ZONES
graph_of_stops = copy.deepcopy(CITY_GRAPH)
list_of_stops = []
place_stops_on_roads(graph_networks_of_polygons, graph_of_stops, list_of_stops) # Adds stops to graph_of_stops, and gets the relevant edges

nid = len(list_of_stops)
if nid in graph_of_stops and graph_of_stops.nodes[nid] == {}:
    graph_of_stops.remove_node(nid)

Starting stop placement for 575 amenity nodes...
[Zone polygon_3304] Area=20364.8m² | Candidate edges=15
  [Zone Stop 0] Edge=(8318090084, 7248441625, 0) | Length=144.51m
    -> Placed at (14.586838, 121.034674) | Split lengths: AB=90.62m, BC=53.90m (orig=144.51m)
  [Zone Stop 1] Edge=(6317818905, 1190359344, 0) | Length=160.73m
    -> Placed at (14.586920, 121.034614) | Split lengths: AB=59.44m, BC=101.29m (orig=160.73m)
  -> Skip edge (7248441625, 243193964, 0): too short (12.60m <= 20m)
  -> Skip edge (243193964, 6317818905, 0): too short (10.86m <= 20m)
  -> Skip edge (6317818906, 243193964, 0): too short (18.87m <= 20m)
  -> Skip edge (243193964, 6317818906, 0): too short (18.87m <= 20m)
  [Zone Stop 2] Edge=(243193964, 1190345315, 0) | Length=360.95m
    -> Placed at (14.587292, 121.037521) | Split lengths: AB=301.00m, BC=59.95m (orig=360.95m)
  [Zone Stop 3] Edge=(6317818906, 323060463, 0) | Length=87.51m
    -> Placed at (14.587889, 121.035426) | Split lengths: AB=82.65m, BC=4.

In [9]:
# Getting all the partner edges (One-way roads that have a partner road that goes the opposite way)
partner_edges = {}
for index, r in filtered_roads_strings.iterrows():
  u1, v1, _key1 = index
  if not r['oneway']:
    continue
  road_geometry = r['geometry']

  possible_partners = filtered_roads_strings.iloc[list(filtered_roads_strings_sindex.intersection(road_geometry.buffer(0.0005).bounds))]
  for idx, point in possible_partners.iterrows():
    u2, v2, _key2 = idx
    if r['edge_id'] != point['edge_id']:
      r_name = r.get('name')
      p_name = point.get('name')
      if point['geometry'].intersects(road_geometry.buffer(0.0005)) and point['oneway'] and r_name == p_name and u1 != v2 and v1 != u2:
        partner_edges[r['edge_id']] = point['edge_id']

In [10]:
# Configuration
MAX_STOPS = [x for x in range(STOP_MIN, STOP_MIN+5)]
MAX_LINES = [x for x in range(N_ROUTES[CITY_NAME] - 5, N_ROUTES[CITY_NAME] + 5)]
WALK_DISTANCE = WALKING_DISTANCES[WD_TO_USE] # WALKING_DISTANCES ->  500, 750, 1000 (changed from 300,550,800)

SEED = 100

run_tag = make_run_tag(WALK_DISTANCE, STOP_MIN, N_ROUTES[CITY_NAME], N_NETWORKS)

TARGET_GEN_DIR = f"../data/initial_route_sets/{CITY_NAME}"
os.makedirs(TARGET_GEN_DIR, exist_ok=True)

file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_population{run_tag}.pkl")
file_path_checkpoint = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_checkpoint_population.pkl")

In [11]:
def worker(i):
  rand = random.Random(SEED + i)
  route_network, route_graph = generate_route_network(
    list_of_stops,
    WALK_DISTANCE,
    rand.choice(MAX_STOPS),
    rand.choice(MAX_LINES),
    graph_of_stops,
    city_area_sum,
  )
  return Network(routes=route_network, graph=route_graph)


initial_population = []

N_WORKERS = 32
ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as exec:
  futures = [exec.submit(worker, i) for i in range(N_NETWORKS)]
  for idx, future in enumerate(as_completed(futures), 1):
    net = future.result()
    initial_population.append(net)
    if idx % (N_NETWORKS//4) == 0 or idx == N_NETWORKS:
      save_data(initial_population, file_path_checkpoint)

save_data(initial_population, file_path)
print(f"Saved {len(initial_population)} networks to {file_path}.")

Target: 8 lines | Candidate stops: 718 | Max attempts: 80Target: 10 lines | Candidate stops: 718 | Max attempts: 100Target: 14 lines | Candidate stops: 718 | Max attempts: 140

Target: 8 lines | Candidate stops: 718 | Max attempts: 80Target: 12 lines | Candidate stops: 718 | Max attempts: 120Target: 8 lines | Candidate stops: 718 | Max attempts: 80Target: 8 lines | Candidate stops: 718 | Max attempts: 80


Target: 10 lines | Candidate stops: 718 | Max attempts: 100Target: 14 lines | Candidate stops: 718 | Max attempts: 140
Target: 6 lines | Candidate stops: 718 | Max attempts: 60Target: 9 lines | Candidate stops: 718 | Max attempts: 90Target: 14 lines | Candidate stops: 718 | Max attempts: 140Target: 5 lines | Candidate stops: 718 | Max attempts: 50Target: 14 lines | Candidate stops: 718 | Max attempts: 140
Target: 8 lines | Candidate stops: 718 | Max attempts: 80Target: 13 lines | Candidate stops: 718 | Max attempts: 130
Target: 7 lines | Candidate stops: 718 | Max attempts: 70

Targe

In [12]:
file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_stops{run_tag}.pkl")
# file_path = os.path.join(TARGET_GEN_DIR, f"{CITY_NAME}_stops.pkl")

save_data(graph_of_stops, file_path)
print(f"Saved stop graph to {file_path}.")

Saved stop graph to ../data/initial_route_sets/Mandaluyong/Mandaluyong_stops_w750_s20_r10_n100.pkl.


### Visualization of Initial Population

In [13]:
# Configuration
OUTPUT_DIR = f"../plots/Initial Route Set Maps/{CITY_NAME}/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

route_network = initial_population[0].routes

# save_route_graph_to_plot(graph_of_stops, route_network, f"{CITY_NAME}_Combined_Routes.html", OUTPUT_DIR)
save_route_graph_to_plot(graph_of_stops, route_network, f"{CITY_NAME}_Combined_Routes{run_tag}.html", OUTPUT_DIR)

Saved Mandaluyong_Combined_Routes_w750_s20_r10_n100.html to: ../plots/Initial Route Set Maps/Mandaluyong/Mandaluyong_Combined_Routes_w750_s20_r10_n100.html


### Visualization of Stops

In [14]:
# stops_map = plot_stops_on_map(list_of_stops_Makati)
# stops_map.save(f"{Makati_map_filepath}stops_map.html")